# Pitwall: fine-tune the router model (free Colab GPU)

Trains a LoRA on Qwen3-0.6B-Base to pick Gearbox's tier (S/M/L) for a prompt, then
exports a quantised GGUF bundle (~400 MB) to download. About 20 minutes.

1. **Runtime -> Change runtime type -> T4 GPU**
2. **Runtime -> Run all**
3. Save `ft-train_old.zip` into `bundles/` of your Pitwall checkout and unzip it there.

The notebook only calls the repo's training script (`pitwall.finetune.train`); its
docstring explains each step and why. A failing `!` command does not stop *Run all*, so
every step checks `_exit_code` and stops the notebook on a failure.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
!git pull

In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [3]:
# The Pitwall commit to train from. "main" = latest; a commit id makes the run repeatable.
COMMIT = "main"
!git clone -q https://github.com/Utsav0706/pitwall.git
%cd pitwall
!git checkout -q {COMMIT} && git log -1 --format='training from %h %s'
assert _exit_code == 0, "checkout failed"

/content/pitwall
training from 55b05ae Colab: remove preinstalled torchao (peft rejects 0.10); use dtype= not torch_dtype


In [4]:
# Colab's own torch is kept (it matches the GPU driver); everything else is pinned.
!pip install -q -e . -r training/requirements-colab.txt
assert _exit_code == 0, "install failed"
# Colab ships torchao 0.10, which peft refuses to load beside (it needs >= 0.16).
# Pitwall doesn't use torchao, so remove it rather than upgrade it against Colab's torch.
!pip uninstall -q -y torchao

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for pitwall (pyproject.toml) ... done


In [5]:
# Rebuild the training data exactly: fetch Gearbox's files from the pinned commit, build
# the pairs, and stop if anything differs from the committed dvc.lock.
!dvc update -q -R data/gearbox
assert _exit_code == 0, "dvc update failed"
!dvc repro -q
assert _exit_code == 0, "dvc repro failed"
!git diff --exit-code --stat
assert _exit_code == 0, "data differs from dvc.lock - stop"
!pitwall data show

source:  https://raw.githubusercontent.com/Utsav0706/gearbox/890e275066c7f9520cbbcb223fb867e9923bac0e/data/v2/prompts.jsonl
prompt:  router:1 (162a384378d3), 0 prompts cut at 3000 chars

file          rows     S    M    L   old hard  sha256
test           119    52   57   10    85   34  5cd073036f1d
train_old      365   172  180   13   365    0  cdfecbea61a4
pool_new       116    37   50   29     0  116  9c2ce5305a6e
train_full     481   209  230   42   365  116  776766f3db15
source:  https://raw.githubusercontent.com/Utsav0706/gearbox/890e275066c7f9520cbbcb223fb867e9923bac0e/data/v2/prompts.jsonl
prompt:  router:1 (162a384378d3), 0 prompts cut at 3000 chars

file          rows     S    M    L   old hard  sha256
test           119    52   57   10    85   34  5cd073036f1d
train_old      365   172  180   13   365    0  cdfecbea61a4
pool_new       116    37   50   29     0  116  9c2ce5305a6e
train_full     481   209  230   42   365  116  776766f3db15


In [6]:
!python -m pitwall.finetune.train     --train data/derived/train_old.jsonl     --out bundles/ft-train_old     --llama-cpp .llama.cpp
assert _exit_code == 0, "training failed - see the output above"

config.json: 100% 727/727 [00:00<00:00, 3.51MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 34.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 69.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 143MB/s]
device cuda; 292 train, 73 validation

model.safetensors: downloading bytes:   9% 109M/1.19G [00:01<00:06, 156MB/s, 8.24MB/s  ] 
model.safetensors: reconstructing file:   3% 33.4M/1.19G [00:01<00:40, 28.8MB/s]
model.safetensors: downloading bytes:  16% 195M/1.19G [00:01<00:05, 198MB/s, 16.0MB/s  ]
model.safetensors: downloading bytes:  19% 232M/1.19G [00:01<00:06, 159MB/s, 20.0MB/s  ]
model.safetensors: downloading bytes:  25% 294M/1.19G [00:02<00:05, 153MB/s, 25.5MB/s  ]
model.safetensors: downloading bytes:  27% 318M/1.19G [00:02<00:05, 169MB/s, 26.2MB/s  ]
model.safetensors: downloading bytes:  33% 397M/1.19G [00:02<00:04, 189MB/s, 32.5MB/s  ]
model.safetensors: downloading bytes:  46% 552M/1.19G [00:03<00:02, 22

AssertionError: training failed - see the output above

In [ ]:
!pitwall bundle check bundles/ft-train_old
assert _exit_code == 0, "bundle check failed"
import shutil

archive = shutil.make_archive("ft-train_old", "zip", "bundles", "ft-train_old")
print(archive)

In [ ]:
# Download. If the browser download stalls (~400 MB), use the Drive cell below instead.
from google.colab import files

files.download("ft-train_old.zip")

In [ ]:
# Optional: copy to Google Drive instead (asks for Drive access).
# from google.colab import drive
# drive.mount("/content/drive")
# !cp ft-train_old.zip /content/drive/MyDrive/